# Fig 3A — Population-mismatch benchmark (data processing)

GTEx-level eQTL simulation (n = 800) with in-sample, ADSP, UKB and 1000 Genomes LD: power / FDR of SuSiE-rRSS against all QC methods across NCP. Bootstrap 95% CIs over replicates.

**Output:** `figure_data/Fig3A.Pop_Mismatch_Benchmark.rds` (+ `.csv`)

In [1]:
source("../../R/utils.R")

data_name <- "Fig3A.Pop_Mismatch_Benchmark"

In [2]:
## ---- assemble per-replicate rows -------------------------------------------
carma         <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_CARMA/summary/result_single_QC_summary.rds")             %>% mutate(qc = "CARMA")
dentist_qc    <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_DEN_SLALOM/summary/result_DEN_SLA_summary_DENTIST_QC.rds")    %>% mutate(qc = "DEN_QC")
dentist_raiss <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_DEN_SLALOM/summary/result_DEN_SLA_summary_DENTIST_RAISS.rds") %>% mutate(qc = "DEN_RAISS")
slalom_raiss  <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_DEN_SLALOM/summary/result_DEN_SLA_summary_SLALOM_RAISS.rds")  %>% mutate(qc = "SLALOM_RAISS")
rsparse_pro   <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_fine_Rsparse_Pro//summary/result_single_QC_summary.rds")           %>% mutate(qc = "Rsparse_Pro")

susie <- readRDS("~/haochen/rss_qc/results/Fig3.Final_power_FDR.rds")
susie_std <- susie %>%
  mutate(panel_raw = str_extract(method, "^[A-Za-z]+"),
         variant   = str_remove(method, "^[A-Za-z]+")) %>%
  mutate(panel = recode(panel_raw, insample = "gtex"),
         qc = case_when(
           variant == ""                     ~ "susie",
           variant == "_mix"                 ~ "susie_eb_mix",
           variant == "_eb_mix_SER_fallback" ~ "susie_eb_SER_fallback",
           TRUE                              ~ NA_character_)) %>%
  filter(!is.na(qc)) %>%
  mutate(method  = panel,
         ncausal = as.integer(str_extract(input_file, "(?<=causal_)\\d+"))) %>%
  select(input_file, method, ncausal, true_var_number, true_set_number,
         total_var_number, total_set_number, total_set_length, qc)

cell_reps <- bind_rows(carma, dentist_qc, dentist_raiss, slalom_raiss, rsparse_pro, susie_std) %>%
  mutate(h2g = str_extract(input_file, "h2g_[0-9.]+"),
         causal_k = total_var_number)          # alias survives group_modify

In [3]:
## ---- bootstrap 95% CIs (boot_cell() in R/utils.R) ---------------------------
n_boot <- 1000
set.seed(1)

summary_ci <- cell_reps %>%
  group_by(qc, method, h2g, total_var_number) %>%
  group_modify(~ boot_cell(.x, B = n_boot)) %>%
  ungroup()

In [4]:
## ---- plot-ready labels / facets ---------------------------------------------
n_gwas <- 800                                   # GTEx eQTL n; NCP = n * h2g

## "SuSiE-rRSS -SER" (EB-mix without the SER fallback) is not in pal_qc, so it
## becomes NA and is dropped by filter(qc != "DENTIST") together with DENTIST.
plot_df <- summary_ci %>%
  mutate(
    method = recode(method,
      gtex = "In-Sample", ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes",
      .default = as.character(method)),
    method = factor(method, levels = c("In-Sample", "ADSP", "UKB", "1000 Genomes")),
    h2g_num = as.numeric(str_remove(h2g, "h2g_")),
    ncp_num = n_gwas * h2g_num,
    ncp = fct_reorder(paste0("NCP = ", formatC(ncp_num, format = "f", digits = 1)), ncp_num),
    qc = recode(qc,
      susie                 = "SuSiE-RSS",
      susie_eb_mix          = "SuSiE-rRSS -SER",
      susie_eb_SER_fallback = "SuSiE-rRSS",
      CARMA                 = "CARMA",
      DEN_QC                = "DENTIST",
      DEN_RAISS             = "DENTIST + RAISS",
      SLALOM_RAISS          = "SLALOM + RAISS",
      Rsparse_Pro           = "RSparsePro",
      .default = as.character(qc)),
    qc = factor(qc, levels = names(pal_qc))
  ) %>% filter(qc != "DENTIST")
stopifnot(!any(is.na(plot_df$qc)))

plot_df <- plot_df %>% filter(ncp != "NCP = 16.0" & ncp != "NCP = 40.0")   # NCPs shown in the figure

save_data(plot_df, data_name)